# MicroReasoner End-to-End Colab Run

This notebook executes setup -> raw data download -> data build -> SFT -> GRPO -> final evaluation -> demo compare.

Run this from the repository root directory.

In [ ]:
from pathlib import Path

if not Path("pyproject.toml").exists() or not Path("src/microreasoner").exists():
    raise RuntimeError(
        "Run this notebook from the MicroReasoner repository root. "
        "If needed: git clone <repo> && cd MicroReasoner"
    )

print("Repository root check passed")

## 0) Environment Setup

In [ ]:
import importlib
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[dev]"])

required_packages = [
    "transformers>=4.46.0",
    "datasets>=2.20.0",
    "accelerate>=0.33.0",
    "peft>=0.11.0",
    "trl>=0.9.6",
    "bitsandbytes>=0.43.0",
    "sentencepiece>=0.2.0",
    "protobuf>=4.25.0",
    "huggingface_hub>=0.24.0",
    "math-verify>=0.6.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *required_packages])

print("Environment setup complete")

## 1) Auto Download Raw Data and Build Canonical JSONL Inputs

This cell downloads GSM8K + MATH, then writes the files expected by `configs/defaults.yaml`:
- `artifacts/raw/openr1_math_subset.jsonl`
- `artifacts/raw/gsm8k_small_mix.jsonl`
- `artifacts/raw/math_small_mix.jsonl`
- `artifacts/datasets/gsm8k_eval.jsonl`
- `artifacts/datasets/math_eval.jsonl`

In [ ]:
import json
import random
import re
from pathlib import Path

from datasets import load_dataset

SEED = 42
TRAIN_GSM8K_TARGET = 2500
TRAIN_MATH_TARGET = 2500
OPENR1_LIKE_TARGET = 5000
EVAL_GSM8K_TARGET = 256
EVAL_MATH_TARGET = 256

rng = random.Random(SEED)
raw_dir = Path("artifacts/raw")
eval_dir = Path("artifacts/datasets")
raw_dir.mkdir(parents=True, exist_ok=True)
eval_dir.mkdir(parents=True, exist_ok=True)


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False))
            handle.write("\n")


def extract_last_number(text: str) -> str | None:
    matches = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?(?:/\d+)?", text)
    if not matches:
        return None
    return matches[-1].replace(",", "")


def extract_gsm8k_answer(answer_text: str) -> str | None:
    if "####" in answer_text:
        candidate = answer_text.split("####")[-1].strip()
        candidate = candidate.replace(",", "")
        if candidate:
            return candidate
    return extract_last_number(answer_text)


def extract_boxed_or_number(solution_text: str) -> str | None:
    boxed = re.search(r"\\boxed\{([^{}]+)\}", solution_text)
    if boxed:
        candidate = boxed.group(1).strip()
        if candidate:
            return candidate
    return extract_last_number(solution_text)


def compact_think(text: str, max_tokens: int = 180) -> str:
    tokens = " ".join(text.strip().split()).split()
    if not tokens:
        return "reasoning omitted"
    return " ".join(tokens[:max_tokens])


def load_math_dataset():
    candidates = [
        ("DigitalLearningGmbH/MATH-lighteval", None),
        ("lighteval/MATH", None),
        ("EleutherAI/hendrycks_math", None),
        ("baber/hendrycks_math", "algebra"),
    ]
    last_error = None
    for repo_id, config_name in candidates:
        try:
            ds = load_dataset(repo_id, config_name) if config_name else load_dataset(repo_id)
            selected = f"{repo_id}:{config_name}" if config_name else repo_id
            print(f"Loaded math dataset: {selected}")
            return ds, selected
        except Exception as exc:
            last_error = exc
            selected = f"{repo_id}:{config_name}" if config_name else repo_id
            print(f"Failed {selected}: {exc}")
    raise RuntimeError(f"Could not load any math dataset candidate. Last error: {last_error}")


def math_question(row: dict) -> str:
    value = row.get("problem")
    if not value:
        value = row.get("question")
    return str(value or "")


def math_solution_text(row: dict) -> str:
    value = row.get("solution")
    if not value:
        value = row.get("answer")
    return str(value or "")


def math_answer_value(row: dict) -> str | None:
    solution_text = math_solution_text(row)
    extracted = extract_boxed_or_number(solution_text)
    if extracted is not None:
        return extracted
    raw_answer = str(row.get("answer", "")).strip()
    if raw_answer:
        return raw_answer
    return None


print("Downloading datasets from Hugging Face...")
gsm8k = load_dataset("openai/gsm8k", "main")
math, math_source_name = load_math_dataset()
math_train_split_name = "train" if "train" in math else next(iter(math.keys()))
math_eval_split_name = (
    "test" if "test" in math else ("validation" if "validation" in math else math_train_split_name)
)
print(f"Using math splits: train={math_train_split_name}, eval={math_eval_split_name}")

gsm_train_rows = []
for idx, row in enumerate(gsm8k["train"]):
    answer = extract_gsm8k_answer(str(row["answer"]))
    if answer is None:
        continue
    think = str(row["answer"]).split("####")[0].strip()
    gsm_train_rows.append(
        {
            "id": f"gsm_train_{idx}",
            "question": str(row["question"]),
            "think": compact_think(think),
            "answer_boxed": answer,
            "benchmark": "gsm8k",
            "source_name": "gsm8k",
            "metadata": {"difficulty": "easy"},
        }
    )

math_train_rows = []
for idx, row in enumerate(math[math_train_split_name]):
    question = math_question(row)
    if question.strip() == "":
        continue
    solution = math_solution_text(row)
    answer = math_answer_value(row)
    if answer is None:
        continue
    difficulty = str(row.get("level") or row.get("difficulty") or "medium").lower()
    think = compact_think(solution) if solution.strip() else "reasoning omitted"
    math_train_rows.append(
        {
            "id": f"math_train_{idx}",
            "question": question,
            "think": think,
            "answer_boxed": answer,
            "benchmark": "math",
            "source_name": math_source_name,
            "metadata": {"difficulty": difficulty},
        }
    )

rng.shuffle(gsm_train_rows)
rng.shuffle(math_train_rows)

gsm_small_mix = gsm_train_rows[: min(TRAIN_GSM8K_TARGET, len(gsm_train_rows))]
math_small_mix = math_train_rows[: min(TRAIN_MATH_TARGET, len(math_train_rows))]

openr1_like_pool = list(gsm_small_mix) + list(math_small_mix)
rng.shuffle(openr1_like_pool)
openr1_math_subset = []
for idx, row in enumerate(openr1_like_pool[: min(OPENR1_LIKE_TARGET, len(openr1_like_pool))]):
    cloned = dict(row)
    cloned["id"] = f"openr1_like_{idx}"
    cloned["source_name"] = "openr1_like_mix"
    openr1_math_subset.append(cloned)

gsm_eval_rows = []
for idx, row in enumerate(gsm8k["test"]):
    answer = extract_gsm8k_answer(str(row["answer"]))
    if answer is None:
        continue
    gsm_eval_rows.append(
        {
            "id": f"gsm_eval_{idx}",
            "question": str(row["question"]),
            "answer": answer,
        }
    )

math_eval_rows = []
for idx, row in enumerate(math[math_eval_split_name]):
    question = math_question(row)
    if question.strip() == "":
        continue
    answer = math_answer_value(row)
    if answer is None:
        continue
    math_eval_rows.append(
        {
            "id": f"math_eval_{idx}",
            "question": question,
            "answer": answer,
        }
    )

rng.shuffle(gsm_eval_rows)
rng.shuffle(math_eval_rows)
gsm_eval_rows = gsm_eval_rows[: min(EVAL_GSM8K_TARGET, len(gsm_eval_rows))]
math_eval_rows = math_eval_rows[: min(EVAL_MATH_TARGET, len(math_eval_rows))]

write_jsonl(raw_dir / "openr1_math_subset.jsonl", openr1_math_subset)
write_jsonl(raw_dir / "gsm8k_small_mix.jsonl", gsm_small_mix)
write_jsonl(raw_dir / "math_small_mix.jsonl", math_small_mix)
write_jsonl(eval_dir / "gsm8k_eval.jsonl", gsm_eval_rows)
write_jsonl(eval_dir / "math_eval.jsonl", math_eval_rows)

print("Raw/eval files written:")
print("-", raw_dir / "openr1_math_subset.jsonl", "rows=", len(openr1_math_subset))
print("-", raw_dir / "gsm8k_small_mix.jsonl", "rows=", len(gsm_small_mix))
print("-", raw_dir / "math_small_mix.jsonl", "rows=", len(math_small_mix))
print("-", eval_dir / "gsm8k_eval.jsonl", "rows=", len(gsm_eval_rows))
print("-", eval_dir / "math_eval.jsonl", "rows=", len(math_eval_rows))

## 2) Build Datasets (data build-sft, data build-rl)

In [ ]:
import json
import subprocess
import sys
from pathlib import Path


def run_cmd(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


run_cmd([
    sys.executable,
    "-m",
    "microreasoner",
    "data",
    "build-sft",
    "--config",
    "configs/defaults.yaml",
    "--source-dir",
    "artifacts/raw",
    "--run-id",
    "colab-build-sft",
])

run_cmd([
    sys.executable,
    "-m",
    "microreasoner",
    "data",
    "build-rl",
    "--config",
    "configs/defaults.yaml",
    "--source-dir",
    "artifacts/raw",
    "--run-id",
    "colab-build-rl",
])

sft_summary = json.loads(Path("artifacts/runs/colab-build-sft/summary.json").read_text(encoding="utf-8"))
rl_summary = json.loads(Path("artifacts/runs/colab-build-rl/summary.json").read_text(encoding="utf-8"))
SFT_MANIFEST = sft_summary["artifacts"]["dataset_manifest_path"]
RL_MANIFEST = rl_summary["artifacts"]["dataset_manifest_path"]

print("SFT_MANIFEST=", SFT_MANIFEST)
print("RL_MANIFEST=", RL_MANIFEST)

## 3) Train SFT (train sft)

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

SFT_MAX_STEPS = 300

cmd = [
    sys.executable,
    "-m",
    "microreasoner",
    "train",
    "sft",
    "--config",
    "configs/defaults.yaml",
    "--dataset-manifest",
    SFT_MANIFEST,
    "--run-id",
    "colab-sft",
    "--max-steps",
    str(SFT_MAX_STEPS),
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

sft_ckpt = json.loads(Path("artifacts/runs/colab-sft/checkpoints.json").read_text(encoding="utf-8"))
SFT_BEST_CHECKPOINT = sft_ckpt["best"]
print("SFT_BEST_CHECKPOINT=", SFT_BEST_CHECKPOINT)

## 4) Train GRPO (train grpo)

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

GRPO_MAX_STEPS = 300

cmd = [
    sys.executable,
    "-m",
    "microreasoner",
    "train",
    "grpo",
    "--config",
    "configs/defaults.yaml",
    "--dataset-manifest",
    RL_MANIFEST,
    "--init-checkpoint",
    SFT_BEST_CHECKPOINT,
    "--run-id",
    "colab-grpo",
    "--max-steps",
    str(GRPO_MAX_STEPS),
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

grpo_ckpt = json.loads(Path("artifacts/runs/colab-grpo/checkpoints.json").read_text(encoding="utf-8"))
GRPO_BEST_CHECKPOINT = grpo_ckpt["best"]
print("GRPO_BEST_CHECKPOINT=", GRPO_BEST_CHECKPOINT)

## 5) Download Base Checkpoint Locally

Final evaluation runner expects local checkpoint paths for Base/SFT/GRPO.

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download

BASE_MODEL_ID = "Qwen/Qwen2.5-Math-1.5B-Instruct"
BASE_CHECKPOINT_DIR = Path("checkpoints/base")
BASE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=BASE_MODEL_ID,
    local_dir=str(BASE_CHECKPOINT_DIR),
    local_dir_use_symlinks=False,
)
print("BASE_CHECKPOINT_DIR=", BASE_CHECKPOINT_DIR)

## 6) Locked Final Evaluation

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "scripts/run_final_evaluation.py",
    "--config",
    "configs/defaults.yaml",
    "--dataset-dir",
    "artifacts/datasets",
    "--base-checkpoint",
    "checkpoints/base",
    "--sft-checkpoint",
    SFT_BEST_CHECKPOINT,
    "--grpo-checkpoint",
    GRPO_BEST_CHECKPOINT,
    "--output-root",
    "artifacts/final_eval",
    "--report-dir",
    "reports",
    "--mode",
    "real",
    "--strict-claims",
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

## 7) Side-by-Side Demo Report

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "scripts/run_demo_compare.py",
    "--config",
    "configs/defaults.yaml",
    "--dataset-dir",
    "artifacts/datasets",
    "--base-checkpoint",
    "checkpoints/base",
    "--sft-checkpoint",
    SFT_BEST_CHECKPOINT,
    "--grpo-checkpoint",
    GRPO_BEST_CHECKPOINT,
    "--output-root",
    "artifacts/demo",
    "--output-file",
    "reports/demo_compare.md",
    "--mode",
    "real",
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

print("Generated reports:")
print("- reports/final_metrics.json")
print("- reports/final_report.md")
print("- reports/error_analysis.md")
print("- reports/demo_compare.md")